## Topic Modeling with BERTopic (text-embedding-3-small)

Replaces the raw-embedding k-means/HDBSCAN sweep with BERTopic for W3 ground truth construction. Run per category, matching the earlier sweep's structure, one BERTopic model per category rather than one global model, for the same reason as before: your categories are broad and heterogeneous, and category-wise reporting is how the rest of this benchmark is structured.

**Why the switch**: the earlier k-means sweep produced silhouette scores around 0.01, essentially indistinguishable from random partitioning. This is a known failure mode of k-means/HDBSCAN on raw high-dimensional text embeddings (1536-dim here): distances concentrate in high dimensions, so density- or centroid-based clustering on the raw space struggles to find real separation. BERTopic's pipeline is UMAP (dimensionality reduction to a low-dim space where local neighborhood structure survives) into HDBSCAN (density clustering in that reduced space) into c-TF-IDF (keyword extraction per cluster). The UMAP step is the actual fix, it's what makes the downstream clustering step viable, HDBSCAN by itself was already tried and didn't solve the problem.

**Embeddings are reused, not regenerated**: BERTopic normally embeds documents itself using a sentence-transformer model. That's bypassed here, your existing `text-embedding-3-small` vectors are passed in directly via BERTopic's `embeddings` parameter. This keeps every clustering method in this project (k-means, HDBSCAN, now BERTopic) operating on the exact same underlying vectors, so differences in results reflect differences in clustering algorithm, not differences in embedding model.

**Outputs**:
1. Per-category topic assignments (chunk_id -> topic_id), same shape as the earlier cluster-assignment files
2. Per-category topic summary CSV (topic id, size, top keywords, representative titles/texts) for direct use in the paper
3. Static matplotlib figures (topic size bar chart, 2D UMAP scatter colored by topic) sized for IEEE two-column format
4. Interactive Plotly HTML files (BERTopic's built-in topic/document visualizations) for your own exploration, not intended for the paper itself

### Install dependencies (Colab only, skip if already installed locally)

In [1]:
# Uncomment if running in Colab or a fresh environment
# !pip install bertopic umap-learn hdbscan huggingface_hub plotly matplotlib pandas -q

### Configuration

In [2]:
import os

# --- Hugging Face repo (shared corpus) ---
REPO_ID = "Sakhiur/empirical-rag-paradigm-benchmark"
REPO_TYPE = "dataset"

# --- Which model's embeddings to use ---
EMBEDDING_MODEL = "text-embedding-3-small"

# --- The 10 target categories ---
ALL_CATEGORIES = [
    "cs.AI", "cs.LG", "cs.IR", "cs.DB", "cs.SE",       # yours
    "cs.CV", "cs.CL", "cs.NE", "cs.DC", "cs.CR",       # collaborator's
]

# === BEST PARAMETERS PER CATEGORY, ABSTRACTS (from full 72-config grid sweep) ===
# Selected as: minimize outlier_pct subject to n_topics landing in a usable 15-60 range
# for the 50-query stratified sampling budget. Full sweep results, not single-config picks.
BEST_PARAMS_ABSTRACTS = {
    "cs.AI":  {"min_cluster_size": 50, "umap_n_neighbors": 10, "umap_n_components": 5, "min_samples": 5},
    "cs.LG":  {"min_cluster_size": 80, "umap_n_neighbors": 10, "umap_n_components": 5, "min_samples": 5},
    "cs.CV":  {"min_cluster_size": 80, "umap_n_neighbors": 10, "umap_n_components": 5, "min_samples": 5},
    "cs.CL":  {"min_cluster_size": 80, "umap_n_neighbors": 10, "umap_n_components": 5, "min_samples": 5},
    "cs.IR":  {"min_cluster_size": 30, "umap_n_neighbors": 15, "umap_n_components": 5, "min_samples": 10},
    "cs.SE":  {"min_cluster_size": 50, "umap_n_neighbors": 10, "umap_n_components": 5, "min_samples": 5},
    "cs.DB":  {"min_cluster_size": 30, "umap_n_neighbors": 10, "umap_n_components": 5, "min_samples": 10},
    "cs.NE":  {"min_cluster_size": 30, "umap_n_neighbors": 30, "umap_n_components": 10, "min_samples": 5},
    "cs.DC":  {"min_cluster_size": 30, "umap_n_neighbors": 10, "umap_n_components": 5, "min_samples": 5},
    "cs.CR":  {"min_cluster_size": 30, "umap_n_neighbors": 10, "umap_n_components": 5, "min_samples": 5},
}

# === BEST PARAMETERS PER CATEGORY, PDF CHUNKS ===
BEST_PARAMS_PDF = {
    "cs.AI":  {"min_cluster_size": 50, "umap_n_neighbors": 10, "umap_n_components": 5, "min_samples": 5},
    "cs.LG":  {"min_cluster_size": 50, "umap_n_neighbors": 10, "umap_n_components": 5, "min_samples": 5},
    "cs.CV":  {"min_cluster_size": 30, "umap_n_neighbors": 10, "umap_n_components": 5, "min_samples": 5},
    "cs.CL":  {"min_cluster_size": 30, "umap_n_neighbors": 10, "umap_n_components": 5, "min_samples": 5},
    "cs.IR":  {"min_cluster_size": 30, "umap_n_neighbors": 10, "umap_n_components": 5, "min_samples": 5},
    "cs.SE":  {"min_cluster_size": 30, "umap_n_neighbors": 10, "umap_n_components": 5, "min_samples": 5},
    "cs.DB":  {"min_cluster_size": 30, "umap_n_neighbors": 10, "umap_n_components": 5, "min_samples": 5},
    "cs.NE":  {"min_cluster_size": 30, "umap_n_neighbors": 10, "umap_n_components": 5, "min_samples": 5},
    "cs.DC":  {"min_cluster_size": 30, "umap_n_neighbors": 10, "umap_n_components": 5, "min_samples": 5},
    "cs.CR":  {"min_cluster_size": 30, "umap_n_neighbors": 10, "umap_n_components": 5, "min_samples": 5},
}


def get_best_params(category: str, corpus_type: str) -> dict:
    """Returns the correct parameter set for the given category AND corpus type.
    Previously the run loop used one shared BEST_PARAMS dict for both abstracts and
    pdf_chunks, silently applying abstract-tuned parameters to the PDF-chunk population
    too, this function is the fix: it routes to the corpus-type-specific dict instead.
    """
    if corpus_type == "abstracts":
        return BEST_PARAMS_ABSTRACTS[category]
    elif corpus_type == "pdf_chunks":
        return BEST_PARAMS_PDF[category]
    else:
        raise ValueError(f"Unknown corpus_type: {corpus_type}")


# --- Reproducibility ---
RANDOM_STATE = 42

# --- Visualization settings ---
IEEE_COLUMN_WIDTH_INCHES = 3.5
IEEE_FIGURE_DPI = 300
N_TOP_KEYWORDS_DISPLAY = 8
N_REPRESENTATIVE_DOCS = 3

# --- Local paths ---
PROJECT_ROOT = ".."
BERTOPIC_DIR = os.path.join(PROJECT_ROOT, "data", "bertopic", EMBEDDING_MODEL)
FIGURES_DIR = os.path.join(PROJECT_ROOT, "results", "figures", "bertopic")
INTERACTIVE_DIR = os.path.join(BERTOPIC_DIR, "interactive")

for corpus_type in ["abstracts", "pdf_chunks"]:
    os.makedirs(os.path.join(BERTOPIC_DIR, corpus_type), exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(INTERACTIVE_DIR, exist_ok=True)

FORCE_RECOMPUTE = True


### Why 15-60 topics with under 35% outliers, and why abstracts and PDF chunks are tuned separately

This range is well-balanced for the downstream task of generating 50 queries per category via stratified sampling:

- 15-40 topics: ideal, allows 1-4 queries per topic on average.
- Up to 60 topics: still workable, most topics get at least 1 query.
- Under 35% outliers: a reasonable share of documents are meaningfully clustered rather than discarded.

Fewer than 15 topics makes clusters too coarse for W3 ground truth. More than 60 risks many topics getting zero queries under the 50-sample budget, the same failure mode the earlier TVD analysis diagnosed for the raw min_cluster_size=10 run.

**Abstracts vs PDF chunks need separate parameter sets, not one shared dict.** Abstracts are paper-level (one embedding per paper), PDF chunks are page-level (one embedding per page), so for the same category the PDF-chunk population is a different size and shape than the abstract population, e.g. cs.AI has 7,369 abstracts but only 2,060 PDF page-chunks in the sampled corpus. A min_cluster_size tuned for 7,369 abstracts has no reason to produce a comparable topic count on a 2,060-chunk population, so BEST_PARAMS_ABSTRACTS and BEST_PARAMS_PDF are maintained as two separate dicts, routed through get_best_params(category, corpus_type), rather than one dict silently reused for both corpus types.

### Authenticate with Hugging Face

In [3]:
from huggingface_hub import whoami, login

try:
    user_info = whoami()
    print(f"Authenticated as: {user_info['name']}")
except Exception:
    print("Not authenticated yet, prompting for token...")
    login()
    user_info = whoami()
    print(f"Authenticated as: {user_info['name']}")

c:\project\empirical-rag-paradigm-benchmark\benchmarking-research\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Authenticated as: Sakhiur


### Loader functions

Same join pattern as the earlier clustering notebook: embeddings and source records are loaded separately and joined on `chunk_id`, since embeddings alone carry no human-readable text and BERTopic needs both (embeddings for clustering, raw text for c-TF-IDF keyword extraction and for the representative-document display).

In [4]:
import json
import numpy as np
from huggingface_hub import hf_hub_download


def load_embeddings(category: str, corpus_type: str) -> dict[str, list[float]]:
    repo_path = f"embeddings/{EMBEDDING_MODEL}/{corpus_type}/{category}.jsonl"
    try:
        local_path = hf_hub_download(repo_id=REPO_ID, repo_type=REPO_TYPE, filename=repo_path)
    except Exception as e:
        print(f"  [SKIP] {repo_path} not found on Hub ({e})")
        return {}

    result = {}
    with open(local_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)
            result[record["chunk_id"]] = record["embedding"]
    return result


def load_source_records(category: str, corpus_type: str) -> dict[str, dict]:
    if corpus_type == "abstracts":
        repo_path = f"abstracts/by_category/{category}.jsonl"
    else:
        repo_path = f"pdf_chunks/{category}.jsonl"

    try:
        local_path = hf_hub_download(repo_id=REPO_ID, repo_type=REPO_TYPE, filename=repo_path)
    except Exception:
        return {}

    result = {}
    with open(local_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)
            if corpus_type == "abstracts":
                chunk_id = record["id"]
            else:
                chunk_id = f"{record['paper_id']}_p{record['page_num']}"
            result[chunk_id] = record
    return result


def get_display_text(record: dict, corpus_type: str) -> str:
    """Text fed to BERTopic's c-TF-IDF step and shown in outputs."""
    if corpus_type == "abstracts":
        return record["text"]  # title + abstract, per existing schema
    else:
        return record["text"]  # extracted page text

### BERTopic model builder

**Why `min_cluster_size` is now a function parameter, not a config constant**: a single fixed value (e.g. 10) applied to every category regardless of size produced wildly different cluster counts, from ~30 clusters on a 1,000-document category to 200+ clusters on an 8,500-document category. This broke downstream stratified sampling, since a fixed sample budget of 50 cannot proportionally represent 200+ strata without most of them rounding to zero allocation. Scaling min_cluster_size with population (via `get_min_cluster_size` in the config cell) keeps cluster count in a comparable range across categories, which is both more defensible for BERTopic's own topic-coherence purpose (a 10-document 'topic' out of 8,500 documents is arguably too fine-grained to be a meaningful sub-theme anyway) and directly fixes the downstream sampling representativeness problem.

**Why explicit UMAP and HDBSCAN sub-model objects rather than BERTopic's defaults**: BERTopic accepts pre-configured `umap_model` and `hdbscan_model` objects, which is used here for two reasons. First, it lets `random_state` be fixed on UMAP explicitly, UMAP is stochastic, and an unfixed seed means re-running this notebook could shift topic boundaries and IDs, breaking reproducibility of anything downstream (W3 queries referencing topic N would silently refer to a different topic on a re-run). Second, it makes every hyperparameter visible in this notebook's config cell rather than hidden inside BERTopic's internal defaults, so the methodology section can cite exact values rather than "we used BERTopic with default settings."

**Why `calculate_probabilities=False`**: BERTopic can compute soft cluster-membership probabilities per document, which is useful for some applications but roughly doubles runtime and isn't needed here, chunk_ids get a single hard topic assignment for W3 ground truth, matching how the k-means/HDBSCAN outputs were structured.

In [5]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN


def build_bertopic_model(params: dict) -> BERTopic:
    """Build BERTopic model using category-specific best parameters."""
    umap_model = UMAP(
        n_neighbors=params["umap_n_neighbors"],
        n_components=params["umap_n_components"],
        min_dist=0.0,
        metric="cosine",
        random_state=RANDOM_STATE,
    )
    hdbscan_model = HDBSCAN(
        min_cluster_size=params["min_cluster_size"],
        min_samples=params["min_samples"],
        metric="euclidean",
        cluster_selection_method="eom",
        prediction_data=True,
    )
    return BERTopic(
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        calculate_probabilities=False,
        verbose=False,
    )

### Run BERTopic per category, per corpus type

`min_cluster_size` is computed per category from its population size via `get_min_cluster_size`, not read from a single fixed config value, this is what actually fixes the over-fragmentation problem observed in downstream sampling. `fit_transform` is called with `embeddings=vectors` explicitly, this is what bypasses BERTopic's default internal embedding step and forces it to cluster your existing `text-embedding-3-small` vectors instead. Topic `-1` is BERTopic's convention for outlier documents that didn't fit any topic confidently, same meaning as HDBSCAN's noise label in the earlier k-means/HDBSCAN notebook, kept as-is rather than forced into a topic, since forcing outliers into a nearby topic would quietly degrade that topic's coherence.

In [6]:
all_models = {}
all_results = {}

for corpus_type in ["abstracts", "pdf_chunks"]:
    for category in ALL_CATEGORIES:
        print(f"\n=== {corpus_type} / {category} ===")

        embeddings_dict = load_embeddings(category, corpus_type)
        if not embeddings_dict:
            continue

        source_records = load_source_records(category, corpus_type)
        if not source_records:
            print(f"  [SKIP] No source records")
            continue

        chunk_ids = [cid for cid in embeddings_dict.keys() if cid in source_records]
        vectors = np.array([embeddings_dict[cid] for cid in chunk_ids], dtype=np.float32)
        texts = [get_display_text(source_records[cid], corpus_type) for cid in chunk_ids]

        params = get_best_params(category, corpus_type)
        print(f"  {len(chunk_ids)} documents | corpus_type={corpus_type} | Using best params: mcs={params['min_cluster_size']}")

        model = build_bertopic_model(params)
        topics, _ = model.fit_transform(texts, embeddings=vectors)

        topic_info = model.get_topic_info()
        n_topics = len(topic_info[topic_info["Topic"] != -1])
        n_outliers = int(np.sum(np.array(topics) == -1))

        print(f"  {n_topics} topics found, {n_outliers}/{len(chunk_ids)} outliers ({n_outliers/len(chunk_ids)*100:.1f}%)")

        all_models[(corpus_type, category)] = model
        all_results[(corpus_type, category)] = {
            "chunk_ids": chunk_ids,
            "topics": topics,
            "vectors": vectors,
            "texts": texts,
            "params_used": params,
        }


=== abstracts / cs.AI ===
  7369 documents | corpus_type=abstracts | Using best params: mcs=50
  36 topics found, 2082/7369 outliers (28.3%)

=== abstracts / cs.LG ===
  10746 documents | corpus_type=abstracts | Using best params: mcs=80
  30 topics found, 1923/10746 outliers (17.9%)

=== abstracts / cs.IR ===
  3076 documents | corpus_type=abstracts | Using best params: mcs=30
  13 topics found, 528/3076 outliers (17.2%)

=== abstracts / cs.DB ===
  1532 documents | corpus_type=abstracts | Using best params: mcs=30
  18 topics found, 398/1532 outliers (26.0%)

=== abstracts / cs.SE ===
  2281 documents | corpus_type=abstracts | Using best params: mcs=50
  13 topics found, 308/2281 outliers (13.5%)

=== abstracts / cs.CV ===
  14163 documents | corpus_type=abstracts | Using best params: mcs=80
  55 topics found, 3775/14163 outliers (26.7%)

=== abstracts / cs.CL ===
  12418 documents | corpus_type=abstracts | Using best params: mcs=80
  40 topics found, 3071/12418 outliers (24.7%)

==

### Save topic assignments locally

Same JSONL shape as the k-means/HDBSCAN output, `chunk_id` + assigned topic, so downstream QA-generation code doesn't need to know which clustering method produced the label, it just reads `bertopic_topic`.

In [7]:
for (corpus_type, category), result in all_results.items():
    out_path = os.path.join(BERTOPIC_DIR, corpus_type, f"{category}.jsonl")

    if os.path.exists(out_path) and not FORCE_RECOMPUTE:
        print(f"[SKIP] {corpus_type}/{category}: already saved at {out_path}")
        continue

    with open(out_path, "w", encoding="utf-8") as f:
        for cid, topic in zip(result["chunk_ids"], result["topics"]):
            f.write(json.dumps({"chunk_id": cid, "bertopic_topic": int(topic)}) + "\n")

    print(f"Saved {out_path}")

Saved ..\data\bertopic\text-embedding-3-small\abstracts\cs.AI.jsonl
Saved ..\data\bertopic\text-embedding-3-small\abstracts\cs.LG.jsonl
Saved ..\data\bertopic\text-embedding-3-small\abstracts\cs.IR.jsonl
Saved ..\data\bertopic\text-embedding-3-small\abstracts\cs.DB.jsonl
Saved ..\data\bertopic\text-embedding-3-small\abstracts\cs.SE.jsonl
Saved ..\data\bertopic\text-embedding-3-small\abstracts\cs.CV.jsonl
Saved ..\data\bertopic\text-embedding-3-small\abstracts\cs.CL.jsonl
Saved ..\data\bertopic\text-embedding-3-small\abstracts\cs.NE.jsonl
Saved ..\data\bertopic\text-embedding-3-small\abstracts\cs.DC.jsonl
Saved ..\data\bertopic\text-embedding-3-small\abstracts\cs.CR.jsonl
Saved ..\data\bertopic\text-embedding-3-small\pdf_chunks\cs.AI.jsonl
Saved ..\data\bertopic\text-embedding-3-small\pdf_chunks\cs.LG.jsonl
Saved ..\data\bertopic\text-embedding-3-small\pdf_chunks\cs.IR.jsonl
Saved ..\data\bertopic\text-embedding-3-small\pdf_chunks\cs.DB.jsonl
Saved ..\data\bertopic\text-embedding-3-smal

### Generate topic summary CSVs

One CSV per category, one row per topic (excluding outliers), columns: `topic_id`, `size`, `top_keywords` (c-TF-IDF derived), `representative_titles` (or text previews for PDF chunks). This is the artifact meant to go directly into the paper, either as a table itself or as the source data for a table, rather than needing to be manually assembled from BERTopic's internal objects.

In [8]:
import pandas as pd

for (corpus_type, category), result in all_results.items():
    model = all_models[(corpus_type, category)]
    chunk_ids = result["chunk_ids"]
    topics = result["topics"]
    texts = result["texts"]

    csv_path = os.path.join(BERTOPIC_DIR, corpus_type, f"{category}_topic_summary.csv")
    if os.path.exists(csv_path) and not FORCE_RECOMPUTE:
        print(f"[SKIP] {corpus_type}/{category}: CSV already exists at {csv_path}")
        continue

    topic_info = model.get_topic_info()
    topic_info = topic_info[topic_info["Topic"] != -1].sort_values("Topic")

    rows = []
    for _, row in topic_info.iterrows():
        topic_id = row["Topic"]
        keywords = [kw for kw, _ in model.get_topic(topic_id)[:N_TOP_KEYWORDS_DISPLAY]]

        member_indices = [i for i, t in enumerate(topics) if t == topic_id]
        sample_indices = member_indices[:N_REPRESENTATIVE_DOCS]
        representative_texts = [texts[i][:120].replace("\n", " ") for i in sample_indices]

        rows.append({
            "topic_id": topic_id,
            "size": row["Count"],
            "top_keywords": ", ".join(keywords),
            "representative_examples": " | ".join(representative_texts),
        })

    df = pd.DataFrame(rows)
    df.to_csv(csv_path, index=False)
    print(f"Saved {csv_path} ({len(df)} topics)")

Saved ..\data\bertopic\text-embedding-3-small\abstracts\cs.AI_topic_summary.csv (36 topics)
Saved ..\data\bertopic\text-embedding-3-small\abstracts\cs.LG_topic_summary.csv (30 topics)
Saved ..\data\bertopic\text-embedding-3-small\abstracts\cs.IR_topic_summary.csv (13 topics)
Saved ..\data\bertopic\text-embedding-3-small\abstracts\cs.DB_topic_summary.csv (18 topics)
Saved ..\data\bertopic\text-embedding-3-small\abstracts\cs.SE_topic_summary.csv (13 topics)
Saved ..\data\bertopic\text-embedding-3-small\abstracts\cs.CV_topic_summary.csv (55 topics)
Saved ..\data\bertopic\text-embedding-3-small\abstracts\cs.CL_topic_summary.csv (40 topics)
Saved ..\data\bertopic\text-embedding-3-small\abstracts\cs.NE_topic_summary.csv (10 topics)
Saved ..\data\bertopic\text-embedding-3-small\abstracts\cs.DC_topic_summary.csv (11 topics)
Saved ..\data\bertopic\text-embedding-3-small\abstracts\cs.CR_topic_summary.csv (8 topics)
Saved ..\data\bertopic\text-embedding-3-small\pdf_chunks\cs.AI_topic_summary.csv 

### Matplotlib figures for the paper

**Why these two specific figures**: a topic-size bar chart is the standard first figure in BERTopic-based papers, it communicates cluster balance (are topics roughly even, or is one topic dominating) at a glance, which matters for judging whether a category's clustering is usable as W3 ground truth or too skewed to be meaningful. A 2D UMAP scatter colored by topic is the standard second figure, it gives a visual sanity check of separation quality that complements (not replaces) the silhouette-style numeric checks. Both are generated at IEEE single-column width and 300 DPI so they can be dropped into the LaTeX template without resizing artifacts.

**Why a separate 2D UMAP projection, not the same one used for clustering**: the clustering UMAP reduces to 5 dimensions (`UMAP_N_COMPONENTS`), which can't be plotted directly. A fresh UMAP run reducing to exactly 2 dimensions is computed purely for visualization, using the same `random_state` and `n_neighbors` for consistency, but this is a display-only projection and is not what topic assignments were computed from, worth stating explicitly if this figure appears in the paper, so a reader doesn't assume the 2D plot is the literal clustering space.

In [20]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm   # Standard import

plt.rcParams.update({
    "font.size": 8,
    "font.family": "serif",
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 6,
})


def plot_topic_sizes(model: BERTopic, corpus_type: str, category: str, out_path: str) -> None:
    topic_info = model.get_topic_info()
    topic_info = topic_info[topic_info["Topic"] != -1].sort_values("Count", ascending=True)

    labels = [
        f"T{row['Topic']}: " + ", ".join([kw for kw, _ in model.get_topic(row["Topic"])[:3]])
        for _, row in topic_info.iterrows()
    ]

    fig, ax = plt.subplots(figsize=(IEEE_COLUMN_WIDTH_INCHES, max(2.0, 0.25 * len(topic_info))))
    ax.barh(labels, topic_info["Count"], color="#4C72B0")
    ax.set_xlabel("Documents")
    ax.set_title(f"{category} ({corpus_type}): topic sizes")
    fig.tight_layout()
    fig.savefig(out_path, dpi=IEEE_FIGURE_DPI, bbox_inches="tight")
    plt.close(fig)


def plot_umap_scatter(vectors: np.ndarray, topics: list, corpus_type: str, 
                      category: str, out_path: str, umap_n_neighbors: int) -> None:
    """
    2D visualization using the same n_neighbors as the actual clustering for that category.
    """
    reducer_2d = UMAP(
        n_neighbors=umap_n_neighbors,
        n_components=2,
        min_dist=0.0,
        metric="cosine",
        random_state=RANDOM_STATE,
    )
    coords_2d = reducer_2d.fit_transform(vectors)

    topics_arr = np.array(topics)
    unique_topics = sorted(set(topics_arr) - {-1})
    
    # Fixed colormap handling (this was the main bug)
    n_colors = max(len(unique_topics), 1)
    cmap = plt.get_cmap("tab20", n_colors)

    fig, ax = plt.subplots(figsize=(IEEE_COLUMN_WIDTH_INCHES, IEEE_COLUMN_WIDTH_INCHES))

    # Plot outliers
    outlier_mask = topics_arr == -1
    if outlier_mask.any():
        ax.scatter(
            coords_2d[outlier_mask, 0], 
            coords_2d[outlier_mask, 1],
            c="lightgray", s=4, alpha=0.4, label="outlier", linewidths=0,
        )

    # Plot topics
    for i, topic_id in enumerate(unique_topics):
        mask = topics_arr == topic_id
        ax.scatter(
            coords_2d[mask, 0], 
            coords_2d[mask, 1],
            c=[cmap(i)], 
            s=5, 
            alpha=0.7, 
            label=f"T{topic_id}", 
            linewidths=0,
        )

    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    ax.set_title(f"{category} ({corpus_type}): 2D topic projection")
    ax.set_xticks([])
    ax.set_yticks([])
    
    if len(unique_topics) <= 12:
        ax.legend(markerscale=2, loc="best", framealpha=0.8)

    fig.tight_layout()
    fig.savefig(out_path, dpi=IEEE_FIGURE_DPI, bbox_inches="tight")
    plt.close(fig)

### Generate matplotlib figures for all categories

This is the slowest cell in the notebook, the 2D UMAP scatter fits a fresh UMAP model per category purely for visualization, on top of the 5D UMAP already fit during clustering. Expected, not a bug, two different UMAP outputs serve two different purposes (clustering input vs. human-readable plot).

In [21]:
for (corpus_type, category), result in all_results.items():
    model = all_models[(corpus_type, category)]
    params = get_best_params(category, corpus_type)

    size_fig_path = os.path.join(FIGURES_DIR, f"{corpus_type}_{category}_topic_sizes.png")
    scatter_fig_path = os.path.join(FIGURES_DIR, f"{corpus_type}_{category}_umap_scatter.png")

    if not os.path.exists(size_fig_path) or FORCE_RECOMPUTE:
        plot_topic_sizes(model, corpus_type, category, size_fig_path)
        print(f"Saved {size_fig_path}")
    else:
        print(f"[SKIP] {size_fig_path} already exists")

    if not os.path.exists(scatter_fig_path) or FORCE_RECOMPUTE:
        plot_umap_scatter(
            result["vectors"], result["topics"], corpus_type, category, scatter_fig_path,
            umap_n_neighbors=params["umap_n_neighbors"],
        )
        print(f"Saved {scatter_fig_path}")
    else:
        print(f"[SKIP] {scatter_fig_path} already exists")


Saved ..\results\figures\bertopic\abstracts_cs.AI_topic_sizes.png
Saved ..\results\figures\bertopic\abstracts_cs.AI_umap_scatter.png
Saved ..\results\figures\bertopic\abstracts_cs.LG_topic_sizes.png
Saved ..\results\figures\bertopic\abstracts_cs.LG_umap_scatter.png
Saved ..\results\figures\bertopic\abstracts_cs.IR_topic_sizes.png
Saved ..\results\figures\bertopic\abstracts_cs.IR_umap_scatter.png
Saved ..\results\figures\bertopic\abstracts_cs.DB_topic_sizes.png
Saved ..\results\figures\bertopic\abstracts_cs.DB_umap_scatter.png
Saved ..\results\figures\bertopic\abstracts_cs.SE_topic_sizes.png
Saved ..\results\figures\bertopic\abstracts_cs.SE_umap_scatter.png
Saved ..\results\figures\bertopic\abstracts_cs.CV_topic_sizes.png
Saved ..\results\figures\bertopic\abstracts_cs.CV_umap_scatter.png
Saved ..\results\figures\bertopic\abstracts_cs.CL_topic_sizes.png
Saved ..\results\figures\bertopic\abstracts_cs.CL_umap_scatter.png
Saved ..\results\figures\bertopic\abstracts_cs.NE_topic_sizes.png
Sav

### Interactive Plotly exports (exploration only, not for the paper)

Uses BERTopic's built-in `visualize_topics()` (inter-topic distance map) and `visualize_documents()` (document scatter with hover text) methods, saved as standalone HTML files you can open locally and click through. These are for your own review of cluster quality (an interactive, zoomable complement to the static spot-check approach from the earlier notebook), not intended to be embedded in the IEEE paper, which is why they're kept separate from `FIGURES_DIR`.

In [ ]:
for (corpus_type, category), result in all_results.items():
    model = all_models[(corpus_type, category)]

    topics_html_path = os.path.join(INTERACTIVE_DIR, f"{corpus_type}_{category}_topics.html")
    docs_html_path = os.path.join(INTERACTIVE_DIR, f"{corpus_type}_{category}_documents.html")

    if not os.path.exists(topics_html_path) or FORCE_RECOMPUTE:
        try:
            fig = model.visualize_topics()
            fig.write_html(topics_html_path)
            print(f"Saved {topics_html_path}")
        except Exception as e:
            print(f"  [WARN] visualize_topics failed for {corpus_type}/{category}: {e}")

    if not os.path.exists(docs_html_path) or FORCE_RECOMPUTE:
        try:
            fig = model.visualize_documents(result["texts"], embeddings=result["vectors"])
            fig.write_html(docs_html_path)
            print(f"Saved {docs_html_path}")
        except Exception as e:
            print(f"  [WARN] visualize_documents failed for {corpus_type}/{category}: {e}")

### Upload topic assignments and CSVs back to Hugging Face

Figures and interactive HTML files stay local (they belong in your paper submission / repo, not the shared corpus dataset). Only the pipeline-consumable artifacts (topic assignments, summary CSVs) go to the Hub, matching what got uploaded for the earlier k-means/HDBSCAN clustering step.

In [22]:
from huggingface_hub import HfApi

api = HfApi()

for corpus_type in ["abstracts", "pdf_chunks"]:
    local_dir = os.path.join(BERTOPIC_DIR, corpus_type)
    for fname in sorted(os.listdir(local_dir)):
        if not (fname.endswith(".jsonl") or fname.endswith(".csv")):
            continue
        local_path = os.path.join(local_dir, fname)
        repo_path = f"bertopic/{EMBEDDING_MODEL}/{corpus_type}/{fname}"

        api.upload_file(
            path_or_fileobj=local_path,
            path_in_repo=repo_path,
            repo_id=REPO_ID,
            repo_type=REPO_TYPE,
            commit_message=f"Upload BERTopic results for {corpus_type}/{fname} (by {user_info['name']})",
        )
        print(f"Uploaded {repo_path}")

Uploaded bertopic/text-embedding-3-small/abstracts/cs.AI.jsonl
Uploaded bertopic/text-embedding-3-small/abstracts/cs.AI_topic_summary.csv
Uploaded bertopic/text-embedding-3-small/abstracts/cs.CL.jsonl
Uploaded bertopic/text-embedding-3-small/abstracts/cs.CL_topic_summary.csv
Uploaded bertopic/text-embedding-3-small/abstracts/cs.CR.jsonl
Uploaded bertopic/text-embedding-3-small/abstracts/cs.CR_topic_summary.csv
Uploaded bertopic/text-embedding-3-small/abstracts/cs.CV.jsonl
Uploaded bertopic/text-embedding-3-small/abstracts/cs.CV_topic_summary.csv
Uploaded bertopic/text-embedding-3-small/abstracts/cs.DB.jsonl
Uploaded bertopic/text-embedding-3-small/abstracts/cs.DB_topic_summary.csv
Uploaded bertopic/text-embedding-3-small/abstracts/cs.DC.jsonl
Uploaded bertopic/text-embedding-3-small/abstracts/cs.DC_topic_summary.csv
Uploaded bertopic/text-embedding-3-small/abstracts/cs.IR.jsonl
Uploaded bertopic/text-embedding-3-small/abstracts/cs.IR_topic_summary.csv
Uploaded bertopic/text-embedding-3

### Summary

Topic count, outlier rate, and the `min_cluster_size` actually used (now scaled per category, not fixed) per category, the first things worth checking before using these topics as W3 ground truth and before feeding cluster assignments into the stratified sampling notebook. A category with a very high outlier percentage means BERTopic couldn't confidently assign most documents anywhere, which would need `UMAP_N_NEIGHBORS` adjusted or `MIN_CLUSTER_SIZE_DIVISOR` tuned further for that category specifically, rather than treating the topics as usable as-is. `n_topics` should now sit in a comparable range across categories of different sizes; if a category still shows a very high topic count relative to others, that's worth investigating before running stratified sampling against it again.

In [23]:
print(f"{'corpus':<12} {'category':<8} {'n_docs':<8} {'min_clus':<9} {'n_topics':<9} {'n_outliers':<11} {'outlier_%'}")
print("-" * 78)

for (corpus_type, category), result in sorted(all_results.items()):
    topics = result["topics"]
    n_docs = len(topics)
    n_outliers = sum(1 for t in topics if t == -1)
    n_topics = len(set(topics) - {-1})
    outlier_pct = 100 * n_outliers / n_docs if n_docs else 0
    min_clus = result["min_cluster_size_used"]

    print(f"{corpus_type:<12} {category:<8} {n_docs:<8} {min_clus:<9} {n_topics:<9} {n_outliers:<11} {outlier_pct:.1f}%")

print(f"\nFigures saved to: {FIGURES_DIR}")
print(f"Interactive HTML saved to: {INTERACTIVE_DIR}")
print(f"CSVs and topic assignments saved to: {BERTOPIC_DIR}")


corpus       category n_docs   min_clus  n_topics  n_outliers  outlier_%
------------------------------------------------------------------------------


KeyError: 'min_cluster_size_used'